# Wind features demo — shear extrapolation & power curve

Sanity-check [`energy_features/wind.py`](../energy_features/wind.py) on a **synthetic 10 m wind** series:

1. Extrapolate to **100 m** hub height (power-law shear, α = 1/7).
2. Apply air-density correction from constant pressure & temperature.
3. Push through windpowerlib **ModelChain** for the reference **Enercon E-126/4200** turbine.

No OPSD or ERA5 download required. See [docs/methodology.md](../docs/methodology.md).

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from energy_features.wind import (
    DEFAULT_SHEAR_EXPONENT,
    REFERENCE_TURBINE_TYPE,
    add_wind_features,
    extrapolate_wind_speed,
    prepare_modelchain_weather,
    run_reference_turbine_power,
)

RNG = np.random.default_rng(42)
Z_FROM_M = 10.0
HUB_HEIGHT_M = 100.0
SHEAR_ALPHA = DEFAULT_SHEAR_EXPONENT

In [ ]:
# One week of hourly synthetic 10 m wind (m/s) with diurnal + weekly variation
idx = pd.date_range("2020-06-01", periods=7 * 24, freq="h", tz="UTC")
t = np.arange(len(idx), dtype=float)
ws10 = (
    5.0
    + 3.0 * np.sin(2.0 * np.pi * t / 24.0)
    + 1.5 * np.sin(2.0 * np.pi * t / (24.0 * 7.0))
    + RNG.normal(0.0, 0.4, len(idx))
)
ws10 = np.clip(ws10, 0.5, 22.0)

df = pd.DataFrame(
    {
        "ws10": ws10,
        "t2m": 288.15,  # 15 °C in kelvin
        "sp": 101_325.0,  # Pa — near standard atmosphere
    },
    index=idx,
)
df.head()

In [ ]:
# Step 1 — power-law extrapolation 10 m → 100 m
expected_ratio = (HUB_HEIGHT_M / Z_FROM_M) ** SHEAR_ALPHA
ws_hub_manual = extrapolate_wind_speed(df["ws10"], Z_FROM_M, HUB_HEIGHT_M, alpha=SHEAR_ALPHA)

print(f"Shear exponent α = {SHEAR_ALPHA:.4f}")
print(f"Height ratio (100/10) = {HUB_HEIGHT_M / Z_FROM_M:.1f}")
print(f"Expected speed multiplier = {expected_ratio:.3f}")
print(f"Sample 10 m → hub: {df['ws10'].iloc[12]:.2f} → {ws_hub_manual.iloc[12]:.2f} m/s")

pd.DataFrame(
    {
        "ws10_mps": df["ws10"],
        "ws_hub_manual_mps": ws_hub_manual,
        "ratio": ws_hub_manual / df["ws10"],
    }
).describe().loc[["mean", "min", "max"]]

In [ ]:
# Step 2 & 3 — density correction + reference-turbine ModelChain at 100 m hub
out = add_wind_features(
    df,
    wind_speed_col="ws10",
    wind_speed_height_m=Z_FROM_M,
    hub_height_m=HUB_HEIGHT_M,
    shear_exponent=SHEAR_ALPHA,
    pressure_col="sp",
    temperature_col="t2m",
    temperature_unit="K",
)

weather = prepare_modelchain_weather(
    df,
    wind_speed_col="ws10",
    wind_speed_height_m=Z_FROM_M,
    temperature_col="t2m",
    pressure_col="sp",
)
power = run_reference_turbine_power(weather, hub_height=HUB_HEIGHT_M)

out["ref_turbine_power_mw"] = out["ref_turbine_power_w"] / 1e6
power_mw = power / 1e6

print(f"Reference turbine: {REFERENCE_TURBINE_TYPE} @ {HUB_HEIGHT_M:.0f} m hub")
out[
    [
        "ws10",
        "wind_speed_hub_mps",
        "air_density_kg_m3",
        "wind_speed_hub_density_corr_mps",
        "ref_turbine_power_mw",
    ]
].describe()

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

axes[0].plot(out.index, out["ws10"], label="10 m (synthetic)", alpha=0.85)
axes[0].plot(
    out.index, out["wind_speed_hub_mps"], label=f"{HUB_HEIGHT_M:.0f} m hub (power law)", alpha=0.85
)
axes[0].set_ylabel("Wind speed (m/s)")
axes[0].legend(loc="upper right")
axes[0].grid(alpha=0.3)
axes[0].set_title("Vertical extrapolation — hub wind exceeds 10 m measurement")

axes[1].plot(out.index, out["ref_turbine_power_mw"], color="tab:green", label="ModelChain power")
axes[1].set_ylabel("Power (MW)")
axes[1].set_xlabel("Time (UTC)")
axes[1].legend(loc="upper right")
axes[1].grid(alpha=0.3)
axes[1].set_title(f"Reference turbine power ({REFERENCE_TURBINE_TYPE})")

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(out["wind_speed_hub_mps"], out["ref_turbine_power_mw"], s=12, alpha=0.6)
ax.set_xlabel("Hub wind speed (m/s)")
ax.set_ylabel("Power (MW)")
ax.set_title("Power curve sanity — monotonic S-shaped response")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Automated sanity checks
assert (out["wind_speed_hub_mps"] > out["ws10"]).all(), "hub wind should exceed 10 m wind"
assert np.allclose(out["wind_speed_hub_mps"] / out["ws10"], expected_ratio, rtol=1e-9)
assert (out["ref_turbine_power_w"] >= 0.0).all(), "power must be non-negative"
assert out["ref_turbine_power_mw"].max() <= 4.21, "E-126/4200 nominal power is 4.2 MW"
assert power_mw.equals(out["ref_turbine_power_mw"]), (
    "add_wind_features power matches direct ModelChain"
)

# Power should rise with hub wind (allow ties at rated plateau)
sorted_df = out.sort_values("wind_speed_hub_mps")
power_diff = sorted_df["ref_turbine_power_mw"].diff().dropna()
assert (power_diff >= -1e-9).all(), "power curve should be non-decreasing in wind speed"
assert sorted_df["ref_turbine_power_mw"].max() > 0.5, "expect meaningful power in this wind range"

print("All sanity checks passed.")